# Miami Predictive Simulator: Gentrification Pressure

This synthesis notebook aggregates data from the strictly isolated `gold` and `migration` datasets on a read-only basis. It combines Miami-Dade housing prices denominated in Gold, implied state-to-county net migration, and recent All-Cash sales data to formulate a localized 'Gentrification Pressure Score'.

In [ ]:
import os
import pandas as pd
import plotly.express as px

# Read-Only Access to Isolated Datasets
gold_csv = '../../data/gold/miami_housing_vs_gold_1976_2026.csv'
migration_csv = '../../data/migration/migration_data.csv'

try:
    df_gold = pd.read_csv(gold_csv, parse_dates=['Year'], index_col='Year')
    df_migr = pd.read_csv(migration_csv, parse_dates=['Year'], index_col='Year')
    print("Successfully loaded isolated datasets.")
except Exception as e:
    print(f"Error loading data: {e}")

ModuleNotFoundError: No module named 'plotly'

In [ ]:
# Merge Data
df_combined = pd.merge(df_gold, df_migr, left_index=True, right_index=True, how='inner')

# Inject Live-Searched All-Cash Sales Percentage Data for Miami (Miami Association of Realtors)
df_combined['Miami_Cash_Sales_Pct'] = pd.NA
df_combined.loc[df_combined.index.year == 2025, 'Miami_Cash_Sales_Pct'] = 0.422 # 42.2%
df_combined.loc[df_combined.index.year == 2026, 'Miami_Cash_Sales_Pct'] = 0.440 # 44.0%

# To extrapolate a gentrification score across history, let's backfill a conservative base cash baseline 
# (e.g. 25% national avg) so the math doesn't break, and let it ramp up to the recent spikes.
df_combined['Miami_Cash_Sales_Pct'] = df_combined['Miami_Cash_Sales_Pct'].fillna(0.25).astype(float)

df_combined.tail()

NameError: name 'df_gold' is not defined

## Calculating the Gentrification Pressure Score
We define the Gentrification Pressure Score as a synthesis of:
1. **Asset Inflation (HPI in Gold)**: Normalized to highlight periods where real estate aggressively outpaces hard money.
2. **Population Influx**: Scaled Miami-Dade Net Migration.
3. **Financialization Factor**: The percentage of cash sales (acting as a multiplier since cash buyers rapidly price out locals).

In [ ]:
# Normalize components (Min-Max Scaling for vis)
max_hpi = df_combined['HPI_in_Gold'].max()
max_migr = df_combined['Miami_Implied_Net_Migration_Thousands'].max()

df_combined['Norm_HPI_in_Gold'] = df_combined['HPI_in_Gold'] / max_hpi
df_combined['Norm_Migration'] = df_combined['Miami_Implied_Net_Migration_Thousands'] / max_migr

# Calculate Score
df_combined['Gentrification_Pressure_Score'] = (
    df_combined['Norm_HPI_in_Gold'] * 0.4 + 
    df_combined['Norm_Migration'] * 0.4 + 
    df_combined['Miami_Cash_Sales_Pct'] * 0.2
) * 100

df_combined[['Gentrification_Pressure_Score']].tail()

## Geographic Visualization Mapping
Plotting the synthesized pressure score over time for the Miami geographic coordinate.

In [ ]:
# 1. Isolate only 2023 data and ensure strict numeric types for Plotly
df_map = df_combined.reset_index()
df_map = df_map[df_map['Year'].dt.year == 2023].copy()

# 2. Fix the Labeling Logic
# Since we are using Annual data, we label it Q4 as a representative snapshot
df_map['Year_Label'] = 'Q4 2023' 

# 3. Force Numeric Types to prevent the Plotly ValueError
df_map['lat'] = 25.7617
df_map['lon'] = -80.1918
# Explicitly cast to float and ensure NO NaNs remain in the size/color columns
df_map['Gentrification_Pressure_Score'] = df_map['Gentrification_Pressure_Score'].astype(float)
df_map['Gentrification_Pressure_Size'] = df_map['Gentrification_Pressure_Score'].clip(lower=0.1).fillna(0).astype(float)

# 4. Use Density Mapbox for the "Heat Graphic" look (instead of scatter points)
fig = px.density_mapbox(
    df_map,
    lat="lat", 
    lon="lon",     
    z="Gentrification_Pressure_Score", # The "Heat" intensity
    radius=50, # Controls the spread of the heat
    hover_name="Year_Label", 
    hover_data=["HPI_in_Gold", "Miami_Implied_Net_Migration_Thousands", "Miami_Cash_Sales_Pct"],
    mapbox_style="carto-positron",
    zoom=10,
    title="Miami Gentrification Heat Map (Q4 2023)"
)

fig.show()
